# Point Transformer V2 - LAS File Inference

This notebook processes raw .las files and performs segmentation using the trained model.

**Workflow:**
1. Load and preview .las file
2. Extract regions with power lines/vegetation
3. Segment using trained model
4. Visualize results in 3D
5. Export segmented results

**Classes:**
- Class 0: High Vegetation (Green)
- Class 1: Power Lines (Orange) ⚡
- Class 2: Pylons (Blue) 🗼
- Class 3: Others (Gray)

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import laspy
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path
from tqdm import tqdm
import json

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

## 2. Configuration

In [ ]:
# Configuration
CONFIG = {
    # Input
    'las_file': '../data/behms_corridor/segmented_chunk_1.las',  # Path to your .las file
    'checkpoint_path': 'checkpoints/last.ckpt',  # Trained model
    
    # Processing
    'chunk_size': 4096,  # Points per chunk for model inference
    'stride': 2048,  # Overlap between chunks (50% overlap)
    'filter_classes': [5, 10, 11],  # Original classes to keep: High Veg, Power Lines, Pylons
    'max_points': 100000,  # Max points to process (None = all)
    
    # Output
    'output_dir': 'inference_results',
    'save_las': True,  # Save segmented result as .las
    'save_visualizations': True
}

# Class mappings
ORIGINAL_TO_MODEL = {
    5: 0,   # High Vegetation -> Class 0
    10: 1,  # Power Lines -> Class 1
    11: 2,  # Pylons -> Class 2
}

CLASS_NAMES = ['High Vegetation', 'Power Lines', 'Pylons', 'Others']
CLASS_COLORS = ['darkgreen', 'orange', 'blue', 'gray']

# Device setup
if torch.cuda.is_available():
    device = 'cuda'
    print("Using CUDA GPU")
elif torch.backends.mps.is_available():
    device = 'mps'
    print("Using Apple Silicon MPS")
else:
    device = 'cpu'
    print("Using CPU")

# Create output directory
Path(CONFIG['output_dir']).mkdir(exist_ok=True)

print(f"\nConfiguration loaded!")
print(f"Input LAS: {CONFIG['las_file']}")
print(f"Output directory: {CONFIG['output_dir']}")

## 3. Load Model Architecture

In [ ]:
def knn(x, k):
    """Find k nearest neighbors"""
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x**2, dim=1, keepdim=True)
    pairwise_distance = -xx - inner - xx.transpose(2, 1)
    idx = pairwise_distance.topk(k=k, dim=-1)[1]
    return idx


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads=8):
        super().__init__()
        self.d_model = d_model
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.pos_mlp = nn.Sequential(
            nn.Linear(3, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, features, positions, k=16):
        B, N, C = features.shape
        idx = knn(positions.transpose(1, 2), k)
        idx_expanded = idx.unsqueeze(-1).expand(-1, -1, -1, C)
        neighbor_features = torch.gather(
            features.unsqueeze(2).expand(-1, -1, k, -1), 1, idx_expanded
        )
        positions_expanded = positions.unsqueeze(2).expand(-1, -1, k, -1)
        neighbor_positions = torch.gather(
            positions.unsqueeze(2).expand(-1, -1, k, -1), 1,
            idx.unsqueeze(-1).expand(-1, -1, -1, 3)
        )
        relative_pos = positions_expanded - neighbor_positions
        pos_encoding = self.pos_mlp(relative_pos)
        q = self.q(features).unsqueeze(2)
        k = self.k(neighbor_features) + pos_encoding
        v = self.v(neighbor_features) + pos_encoding
        attn = torch.matmul(q, k.transpose(-2, -1)) / (C ** 0.5)
        attn = F.softmax(attn, dim=-1)
        out = torch.matmul(attn, v).squeeze(2)
        features = self.norm1(features + out)
        out = self.ffn(features)
        features = self.norm2(features + out)
        return features


class PointTransformerV2(nn.Module):
    def __init__(self, num_classes=4, d_model=128, num_layers=2):
        super().__init__()
        self.num_classes = num_classes
        self.input_embed = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, d_model)
        )
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model) for _ in range(num_layers)
        ])
        self.seg_head = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, points):
        features = self.input_embed(points)
        for block in self.blocks:
            features = block(features, points)
        logits = self.seg_head(features)
        return logits

print("Model architecture loaded!")

## 4. Load Trained Model

In [ ]:
checkpoint_path = Path(CONFIG['checkpoint_path'])

if not checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint not found at {checkpoint_path}")

print(f"Loading model from {checkpoint_path}...")

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)
hparams = checkpoint['hyper_parameters']

print(f"Model hyperparameters:")
print(f"  num_classes: {hparams['num_classes']}")
print(f"  d_model: {hparams['d_model']}")
print(f"  num_layers: {hparams['num_layers']}")

# Create and load model
model = PointTransformerV2(
    num_classes=hparams['num_classes'],
    d_model=hparams['d_model'],
    num_layers=hparams['num_layers']
)

state_dict = {k.replace('model.', ''): v for k, v in checkpoint['state_dict'].items() if k.startswith('model.')}
model.load_state_dict(state_dict)
model = model.to(device)
model.eval()

print(f"\nModel loaded successfully!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Load and Preview LAS File

In [ ]:
print(f"Loading LAS file: {CONFIG['las_file']}...")
las = laspy.read(CONFIG['las_file'])

# Extract coordinates and classification
points_full = np.vstack((las.x, las.y, las.z)).T.astype(np.float32)
classification_full = las.classification.copy()

print(f"\nLAS File Information:")
print(f"  Total points: {len(points_full):,}")
print(f"  X range: [{points_full[:, 0].min():.2f}, {points_full[:, 0].max():.2f}]")
print(f"  Y range: [{points_full[:, 1].min():.2f}, {points_full[:, 1].max():.2f}]")
print(f"  Z range: [{points_full[:, 2].min():.2f}, {points_full[:, 2].max():.2f}]")

print(f"\nOriginal class distribution:")
unique_classes, counts = np.unique(classification_full, return_counts=True)
for cls, count in zip(unique_classes, counts):
    print(f"  Class {cls}: {count:,} ({100*count/len(classification_full):.2f}%)")

## 6. Filter Relevant Points

In [ ]:
# Filter to keep only relevant classes
if CONFIG['filter_classes']:
    mask = np.isin(classification_full, CONFIG['filter_classes'])
    points = points_full[mask]
    classification = classification_full[mask]
    print(f"\nFiltered to classes {CONFIG['filter_classes']}")
    print(f"  Remaining points: {len(points):,} ({100*len(points)/len(points_full):.2f}%)")
else:
    points = points_full
    classification = classification_full

# Limit number of points if specified
if CONFIG['max_points'] and len(points) > CONFIG['max_points']:
    indices = np.random.choice(len(points), CONFIG['max_points'], replace=False)
    points = points[indices]
    classification = classification[indices]
    print(f"\nDownsampled to {CONFIG['max_points']:,} points")

print(f"\nFinal dataset:")
print(f"  Points to process: {len(points):,}")
unique_classes, counts = np.unique(classification, return_counts=True)
for cls, count in zip(unique_classes, counts):
    print(f"  Class {cls}: {count:,} ({100*count/len(classification):.2f}%)")

## 7. Normalize Points

In [ ]:
# Normalize to zero mean (required by model)
centroid = np.mean(points, axis=0)
points_normalized = points - centroid

print(f"Normalization:")
print(f"  Centroid: [{centroid[0]:.2f}, {centroid[1]:.2f}, {centroid[2]:.2f}]")
print(f"  Normalized range:")
print(f"    X: [{points_normalized[:, 0].min():.2f}, {points_normalized[:, 0].max():.2f}]")
print(f"    Y: [{points_normalized[:, 1].min():.2f}, {points_normalized[:, 1].max():.2f}]")
print(f"    Z: [{points_normalized[:, 2].min():.2f}, {points_normalized[:, 2].max():.2f}]")

## 8. Segment Point Cloud with Model

In [ ]:
def segment_point_cloud(points, model, chunk_size=4096, stride=2048, device='cpu'):
    """
    Segment point cloud using sliding window approach
    """
    n_points = len(points)
    predictions = np.zeros(n_points, dtype=np.int32)
    vote_counts = np.zeros(n_points, dtype=np.int32)
    
    # Create chunks with overlap
    num_chunks = int(np.ceil((n_points - chunk_size) / stride)) + 1
    
    print(f"\nSegmenting {n_points:,} points...")
    print(f"  Chunk size: {chunk_size}")
    print(f"  Stride: {stride}")
    print(f"  Number of chunks: {num_chunks}")
    
    with torch.no_grad():
        for i in tqdm(range(num_chunks), desc="Processing chunks"):
            start_idx = i * stride
            end_idx = min(start_idx + chunk_size, n_points)
            
            # Get chunk
            chunk_points = points[start_idx:end_idx]
            
            # Pad if needed
            if len(chunk_points) < chunk_size:
                padding = np.repeat(chunk_points[-1:], chunk_size - len(chunk_points), axis=0)
                chunk_points = np.vstack([chunk_points, padding])
            
            # Convert to tensor and predict
            chunk_tensor = torch.from_numpy(chunk_points).float().unsqueeze(0).to(device)
            logits = model(chunk_tensor)
            pred = logits.argmax(dim=-1).cpu().numpy()[0]
            
            # Update predictions (voting)
            actual_len = min(chunk_size, end_idx - start_idx)
            predictions[start_idx:end_idx] += pred[:actual_len]
            vote_counts[start_idx:end_idx] += 1
    
    # Average votes
    predictions = (predictions / np.maximum(vote_counts, 1)).astype(np.int32)
    
    return predictions


# Segment the point cloud
predictions = segment_point_cloud(
    points_normalized,
    model,
    chunk_size=CONFIG['chunk_size'],
    stride=CONFIG['stride'],
    device=device
)

print(f"\nSegmentation complete!")
print(f"\nPredicted class distribution:")
unique_pred, counts_pred = np.unique(predictions, return_counts=True)
for cls, count in zip(unique_pred, counts_pred):
    print(f"  {CLASS_NAMES[cls]}: {count:,} ({100*count/len(predictions):.2f}%)")

## 9. 3D Visualization

In [ ]:
def visualize_segmentation_3d(points, predictions, title="Segmentation Result", save_path=None):
    """
    Create 3D visualization of segmented point cloud
    """
    fig = plt.figure(figsize=(20, 8))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    
    # Full view
    ax1 = fig.add_subplot(131, projection='3d')
    ax1.set_title('Full Point Cloud', fontsize=14)
    for cls in range(4):
        mask = predictions == cls
        if mask.sum() > 0:
            size = 15 if cls in [1, 2] else 1
            ax1.scatter(points[mask, 0], points[mask, 1], points[mask, 2],
                       c=CLASS_COLORS[cls], label=CLASS_NAMES[cls], s=size, alpha=0.6)
    ax1.legend()
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y')
    ax1.set_zlabel('Z')
    
    # Power lines only
    ax2 = fig.add_subplot(132, projection='3d')
    ax2.set_title('Power Lines (Orange)', fontsize=14, color='orange', fontweight='bold')
    powerline_mask = predictions == 1
    if powerline_mask.sum() > 0:
        ax2.scatter(points[powerline_mask, 0], points[powerline_mask, 1], 
                   points[powerline_mask, 2], c='orange', s=20, alpha=0.8)
        # Show vegetation as context
        veg_mask = predictions == 0
        if veg_mask.sum() > 0:
            sample_veg = np.random.choice(np.where(veg_mask)[0], 
                                         min(5000, veg_mask.sum()), replace=False)
            ax2.scatter(points[sample_veg, 0], points[sample_veg, 1], 
                       points[sample_veg, 2], c='lightgreen', s=1, alpha=0.2)
    ax2.set_xlabel('X')
    ax2.set_ylabel('Y')
    ax2.set_zlabel('Z')
    
    # Pylons only
    ax3 = fig.add_subplot(133, projection='3d')
    ax3.set_title('Pylons (Blue)', fontsize=14, color='blue', fontweight='bold')
    pylon_mask = predictions == 2
    if pylon_mask.sum() > 0:
        ax3.scatter(points[pylon_mask, 0], points[pylon_mask, 1], 
                   points[pylon_mask, 2], c='blue', s=30, alpha=0.8, marker='^')
        # Show context
        context_mask = (predictions == 0) | (predictions == 1)
        if context_mask.sum() > 0:
            sample_ctx = np.random.choice(np.where(context_mask)[0], 
                                         min(3000, context_mask.sum()), replace=False)
            ctx_colors = ['lightgreen' if predictions[i] == 0 else 'lightsalmon' 
                         for i in sample_ctx]
            ax3.scatter(points[sample_ctx, 0], points[sample_ctx, 1], 
                       points[sample_ctx, 2], c=ctx_colors, s=1, alpha=0.2)
    ax3.set_xlabel('X')
    ax3.set_ylabel('Y')
    ax3.set_zlabel('Z')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Visualization saved to {save_path}")
    
    plt.show()


# Visualize results
if CONFIG['save_visualizations']:
    save_path = f"{CONFIG['output_dir']}/segmentation_3d.png"
else:
    save_path = None

visualize_segmentation_3d(points_normalized, predictions, 
                         title=f"Segmentation: {Path(CONFIG['las_file']).name}",
                         save_path=save_path)

## 10. Export Results as LAS

In [ ]:
if CONFIG['save_las']:
    print("Exporting segmented point cloud as .las file...")
    
    # Create output LAS file
    output_las_path = f"{CONFIG['output_dir']}/segmented_result.las"
    
    # Copy original las header
    header = laspy.LasHeader(point_format=las.header.point_format, version=las.header.version)
    header.scales = las.header.scales
    header.offsets = las.header.offsets
    
    # Create new LAS file
    output_las = laspy.LasData(header)
    
    # Set coordinates (denormalized)
    output_las.x = points[:, 0] + centroid[0]
    output_las.y = points[:, 1] + centroid[1]
    output_las.z = points[:, 2] + centroid[2]
    
    # Map predictions back to original classification scheme
    # Model classes: 0=Veg, 1=PowerLine, 2=Pylon, 3=Others
    # Map to: 5=Veg, 10=PowerLine, 11=Pylon, 1=Others
    model_to_original = {0: 5, 1: 10, 2: 11, 3: 1}
    output_classification = np.array([model_to_original[p] for p in predictions], dtype=np.uint8)
    output_las.classification = output_classification
    
    # Write file
    output_las.write(output_las_path)
    
    print(f"\nSegmented LAS file saved to: {output_las_path}")
    print(f"  Points: {len(output_las.points):,}")
    print(f"  Classes in output:")
    unique_out, counts_out = np.unique(output_classification, return_counts=True)
    class_mapping = {5: 'High Vegetation', 10: 'Power Lines', 11: 'Pylons', 1: 'Others'}
    for cls, count in zip(unique_out, counts_out):
        print(f"    Class {cls} ({class_mapping.get(cls, 'Unknown')}): {count:,}")

## 11. Statistics and Summary

In [ ]:
# Generate statistics
print("\n" + "="*80)
print("SEGMENTATION SUMMARY")
print("="*80)

print(f"\nInput:")
print(f"  LAS file: {CONFIG['las_file']}")
print(f"  Total points in file: {len(points_full):,}")
print(f"  Points processed: {len(points):,}")

print(f"\nModel:")
print(f"  Checkpoint: {CONFIG['checkpoint_path']}")
print(f"  Chunk size: {CONFIG['chunk_size']}")
print(f"  Stride: {CONFIG['stride']}")

print(f"\nResults:")
for i, name in enumerate(CLASS_NAMES):
    count = (predictions == i).sum()
    pct = 100 * count / len(predictions)
    print(f"  {name}: {count:,} points ({pct:.2f}%)")

print(f"\nOutput Files:")
if CONFIG['save_las']:
    print(f"  Segmented LAS: {CONFIG['output_dir']}/segmented_result.las")
if CONFIG['save_visualizations']:
    print(f"  Visualization: {CONFIG['output_dir']}/segmentation_3d.png")

print("="*80)

# Save summary
summary = {
    'input_file': CONFIG['las_file'],
    'total_points_in_file': int(len(points_full)),
    'points_processed': int(len(points)),
    'predictions': {
        CLASS_NAMES[i]: int((predictions == i).sum()) 
        for i in range(4)
    },
    'output_files': {
        'las': f"{CONFIG['output_dir']}/segmented_result.las" if CONFIG['save_las'] else None,
        'visualization': f"{CONFIG['output_dir']}/segmentation_3d.png" if CONFIG['save_visualizations'] else None
    }
}

with open(f"{CONFIG['output_dir']}/summary.json", 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\nSummary saved to {CONFIG['output_dir']}/summary.json")

## Summary

This notebook has:

1. ✅ **Loaded raw .las file** and extracted relevant classes
2. ✅ **Segmented point cloud** using trained Point Transformer V2 model
3. ✅ **Generated 3D visualizations** highlighting power lines and pylons
4. ✅ **Exported results** as new .las file with predicted classifications
5. ✅ **Created summary** statistics and reports

**Key Features:**
- Sliding window approach with overlap for better predictions
- Voting mechanism for overlapping regions
- Maintains original coordinate system
- Focuses on power infrastructure detection

**Output:**
- `segmented_result.las` - Segmented point cloud
- `segmentation_3d.png` - 3D visualization
- `summary.json` - Processing statistics